#Log
* trim each column
* change date column to yyyyMMdd
* sales correction by multiply quantity with absolute price or else let it as before.
* price correction by divided sales with quantity
* rename all column 

In [0]:

import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim, length

In [0]:
#read bronze table

sales= spark.table("workspace.bronze.crm_sales_details")

sales.display()

In [0]:
#null check

null_counts_sales=sales.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in sales.columns])

null_counts_sales.show()

In [0]:
# check if there is negative sales,quantity or price
# 3 negative value and 2 zero
# 5 negative price in sls_price
sales_negative= spark.sql("""
                 SELECT *
                 FROM workspace.bronze.crm_sales_details
                 WHERE sls_sales<=0 OR
                 sls_quantity<=0 OR
                 sls_price<=0
                 """

)

sales_negative.display()

In [0]:
#trim all column with integer value
for field in sales.schema.fields:
    if isinstance(field.dataType, StringType):
        df = sales.withColumn(field.name, trim(col(field.name)))

In [0]:
# change date format in each column to yyyyMMdd
for c in ["sls_order_dt","sls_ship_dt","sls_due_dt"]:
       sales=sales.withColumn(
              c,
              F.when(
                     (F.col(c)==0) | (F.length(col(c))!=8),None).
              otherwise(F.to_date(F.col(c).cast("string"),"yyyyMMdd"))
              )

                      

In [0]:
# sales correction
sales =sales.withColumn("sls_sales",F.when(
                        F.col("sls_sales").isNull() |
                        (F.col("sls_sales")<=0) |
                        (F.col("sls_sales")!=F.col("sls_quantity")*F.abs(F.col("sls_price"))),
                        F.col("sls_quantity")*F.abs(F.col("sls_price")))
                        .otherwise(F.col("sls_sales"))
                        )

# price correction
sales = sales.withColumn("sls_price",F.when(
                         F.col("sls_price").isNull() | (F.col("sls_price") <= 0),
                         F.col("sls_sales") / F.nullif(F.col("sls_quantity"), F.lit(0))
                        ).otherwise(F.col("sls_price"))
)

In [0]:
# validate price and sales <=0
# no negative error
sales_validate_sales= sales[(sales["sls_sales"]<=0) | (sales["sls_price"]<=0)]

sales_validate_sales.display()

In [0]:
rename_map_sales = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales",
    "sls_quantity": "quantity",
    "sls_price": "price"
}

for old_name, new_name in rename_map_sales.items():
    sales = sales.withColumnRenamed(old_name, new_name)

In [0]:
sales.display()

In [0]:
sales.write.mode("overwrite").format('delta').saveAsTable("workspace.silver.crm_sales")

In [0]:
%sql
SELECT *
FROM workspace.silver.crm_sales